# Fine-tune Qwen 3.5 0.8B for local bee decisions

This notebook fine-tunes **unsloth/Qwen3.5-0.8B** with Unsloth supervised fine-tuning and 16-bit LoRA. It trains the model to return the same six-field action JSON used by the bee simulator, from the same system instruction and local observation structure.

The accompanying dataset contains **11,900 balanced synthetic decision turns**: 9,520 train, 1,190 validation and 1,190 test examples. Each prompt uses the simulator's real World.observe schema; the simulator checks every demonstration action against its stated physical preconditions. The SFT loss is applied to the assistant completion.

**Label caveat:** these are reproducible hand-written scenario-teacher decisions, not real bee observations, bee-expert annotations or optimal policies. Training can teach this teacher's heuristics and exact JSON formatting. It cannot establish biological correctness. See DATASET_CARD.md and TRAINING_DATA.md.

For a Google Colab run, select a GPU runtime, run the install cell, upload bee_decision_sft_dataset.zip in the data cell, then run the remaining cells. The base checkpoint is open weight, so no inference API key is used. The notebook saves the LoRA adapter locally.

Default training uses a reproducible 50% action-stratified sample of the training split; validation and test remain complete.


## 1. Install current Qwen 3.5-compatible training libraries

Unsloth's current Qwen 3.5 guidance requires Transformers v5 and recommends bf16/16-bit LoRA over QLoRA for this model family. Installation can take several minutes. Restart the Colab runtime if pip reports that core packages were replaced.

In [ ]:
%pip install --upgrade --quiet unsloth unsloth_zoo "transformers>=5.0.0" "trl>=0.27.0" datasets accelerate peft

## 2. Settings and hardware

The default run uses two epochs, rank-16 LoRA and a 3,072-token cap. Adjust batch size/gradient accumulation for the available GPU. If memory runs short, lower the batch size to 1 and/or sequence cap after checking how many examples this removes.

The original Qwen checkpoint is loaded at 16-bit precision. This is parameter-efficient fine-tuning: LoRA parameters train, while the base weights remain frozen.

In [ ]:
from pathlib import Path
import json
import math
import os
import random
import shutil
import zipfile
from collections import Counter

import torch

SEED = 3407
MODEL_ID = "unsloth/Qwen3.5-0.8B"
DATASET_ARCHIVE = Path("/content/bee_decision_sft_dataset.zip")
DATA_DIR = Path("/content/bee_decision_sft")
OUTPUT_DIR = Path("/content/bee_qwen35_08b_bee_lora")
MAX_SEQ_LENGTH = 3072
NUM_EPOCHS = 2
PER_DEVICE_BATCH_SIZE = 1
GRAD_ACCUMULATION = 8
LORA_RANK = 16
LEARNING_RATE = 1e-4
TRAIN_DATA_FRACTION = 0.5
TRAIN_SAMPLE_SEED = SEED + 1
EVAL_EXAMPLES = 51
MAX_NEW_TOKENS = 160
SAVE_MERGED_16BIT = False

random.seed(SEED)
torch.manual_seed(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime in Colab (Runtime > Change runtime type > GPU).")

gpu = torch.cuda.get_device_name(0)
print(f"GPU: {gpu}")
print(f"CUDA: {torch.version.cuda}; BF16 supported: {torch.cuda.is_bf16_supported()}")
print(f"Model: {MODEL_ID}; train mode: 16-bit base + LoRA (no 4-bit QLoRA).")


## 3. Load the generated train, validation and test transcripts

Upload the supplied ZIP when prompted. You can also set DATASET_ARCHIVE to a mounted Drive path or point DATA_DIR at a directory containing the three JSONL split files and manifest.json.

The files are prompt-completion conversational data: system/user turns contain the simulator prompt and local observation; assistant completion is the target action JSON.

In [ ]:
split_paths = {
    "train": DATA_DIR / "bee_decisions_train.jsonl",
    "validation": DATA_DIR / "bee_decisions_validation.jsonl",
    "test": DATA_DIR / "bee_decisions_test.jsonl",
}
if not all(path.exists() for path in split_paths.values()):
    if not DATASET_ARCHIVE.exists():
        try:
            from google.colab import files
            print("Upload bee_decision_sft_dataset.zip")
            uploaded = files.upload()
            uploaded_zip = next((name for name in uploaded if name.endswith(".zip")), None)
            if uploaded_zip is None:
                raise FileNotFoundError("Please upload bee_decision_sft_dataset.zip")
            DATASET_ARCHIVE = Path("/content") / uploaded_zip
        except ImportError as exc:
            raise FileNotFoundError(
                f"Put bee_decision_sft_dataset.zip in /content or set DATA_DIR. Missing: {split_paths}"
            ) from exc
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(DATASET_ARCHIVE) as archive:
        archive.extractall(DATA_DIR)

manifest = json.loads((DATA_DIR / "manifest.json").read_text())
print(json.dumps({
    "dataset": manifest["dataset_name"],
    "version": manifest["dataset_version"],
    "rows_per_split": manifest["rows_per_split"],
    "action_names": manifest["action_names"],
    "label_origin": manifest["label_origin"],
    "limitations": manifest["limitations"],
}, indent=2))
assert manifest["rows_per_split"] == {"train": 9520, "validation": 1190, "test": 1190}


In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files={key: str(path) for key, path in split_paths.items()},
)
print(dataset)
sample = dataset["train"][0]
print("\nInput messages:")
print(json.dumps(sample["prompt"], ensure_ascii=False, indent=2)[:5000])
print("\nExpected assistant completion:")
print(sample["completion"][0]["content"])
assert [message["role"] for message in sample["prompt"]] == ["system", "user"]
assert len(sample["completion"]) == 1
# Use a seeded, action-stratified sample of the training split only.
# Validation and test stay complete for stable comparisons.
if not 0 < TRAIN_DATA_FRACTION <= 1:
    raise ValueError("TRAIN_DATA_FRACTION must be in (0, 1].")
action_indices = {}
for row_index, completion in enumerate(dataset["train"]["completion"]):
    action_name = json.loads(completion[0]["content"])["action"]
    action_indices.setdefault(action_name, []).append(row_index)
sample_rng = random.Random(TRAIN_SAMPLE_SEED)
selected_indices = []
for action_name in sorted(action_indices):
    indices = action_indices[action_name][:]
    sample_rng.shuffle(indices)
    keep_count = max(1, round(len(indices) * TRAIN_DATA_FRACTION))
    selected_indices.extend(indices[:keep_count])
sample_rng.shuffle(selected_indices)
dataset["train"] = dataset["train"].select(selected_indices)
print(f"Stratified training sample: {len(dataset['train']):,} / {manifest['rows_per_split']['train']:,} rows ({TRAIN_DATA_FRACTION:.0%}).")
print("Validation/test retained:", len(dataset["validation"]), len(dataset["test"]))


## 4. Load Qwen 3.5 and attach trainable LoRA weights

The selected Unsloth checkpoint is the instruct-tuned 0.8B model. The chat tokenizer keeps prompts aligned with the simulator's system/user/assistant turns. The notebook uses the language-model loading path because these bee decisions are text-only.

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=False,
    load_in_16bit=True,
    full_finetuning=False,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Loaded base model: {trainable:,} / {total:,} parameters currently trainable.")


## 5. Check prompt lengths and avoid silent truncation

The simulator observation includes nearby compartment readings and the action guide, so this cell measures the actual tokenized full turns. Examples longer than MAX_SEQ_LENGTH are reported and filtered instead of having their completion silently truncated.

In [ ]:
def measure_lengths(batch):
    lengths = []
    for prompt, completion in zip(batch["prompt"], batch["completion"]):
        turns = prompt + completion
        encoded = tokenizer.apply_chat_template(
            turns, tokenize=True, add_generation_prompt=False
        )
        token_ids = encoded["input_ids"] if hasattr(encoded, "keys") and "input_ids" in encoded else encoded
        if hasattr(token_ids, "shape"):
            token_count = int(token_ids.shape[-1])
        elif token_ids and isinstance(token_ids[0], (list, tuple)):
            token_count = len(token_ids[0])
        else:
            token_count = len(token_ids)
        lengths.append(token_count)
    return {"sequence_length": lengths}

dataset = dataset.map(measure_lengths, batched=True, desc="Measure chat-template token lengths")
length_report = {}
for split_name in ("train", "validation", "test"):
    lengths = dataset[split_name]["sequence_length"]
    over = sum(length > MAX_SEQ_LENGTH for length in lengths)
    length_report[split_name] = {
        "rows": len(lengths),
        "max_tokens": max(lengths),
        "p95_tokens": sorted(lengths)[int(0.95*(len(lengths)-1))],
        "over_cap": over,
    }
print(json.dumps(length_report, indent=2))

dataset = dataset.filter(lambda row: row["sequence_length"] <= MAX_SEQ_LENGTH,
                         desc="Drop overlength examples")
dataset = dataset.remove_columns(["sequence_length"])
assert len(dataset["train"]) > 0 and len(dataset["validation"]) > 0 and len(dataset["test"]) > 0


## 6. Compare the untuned checkpoint on held-out inputs

This small deterministic evaluation establishes a base-model reference. The same held-out examples are used after fine-tuning. Metrics check strict single-object JSON, required fields, basic action/resource types, and exact action-class agreement with the synthetic target. They measure imitation of these labels, not biological correctness.

In [ ]:
FastLanguageModel.for_inference(model)
ALLOWED_ACTIONS = set(manifest["action_names"])
ALLOWED_RESOURCES = {"none", "honey", "nectar", "pollen", "water"}
REQUIRED_FIELDS = {"action", "target", "resource", "amount", "message", "memory"}

def parse_one_json(text):
    decoder = json.JSONDecoder()
    start = len(text) - len(text.lstrip())
    try:
        value, end = decoder.raw_decode(text, start)
    except (json.JSONDecodeError, TypeError):
        return None, False
    return value, not text[end:].strip()

def expected_action(row):
    return json.loads(row["completion"][0]["content"])

def generate_one(row):
    encoded = tokenizer.apply_chat_template(
        row["prompt"], tokenize=True, add_generation_prompt=True,
        return_tensors="pt", return_dict=True,
    )
    encoded = {key: value.to(model.device) for key, value in encoded.items()
               if isinstance(value, torch.Tensor)}
    prompt_tokens = encoded["input_ids"].shape[-1]
    with torch.inference_mode():
        result = model.generate(
            **encoded,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(result[0, prompt_tokens:], skip_special_tokens=True).strip()

def validate_predicted_action(value, strict_one_json):
    if not strict_one_json or not isinstance(value, dict) or set(value) != REQUIRED_FIELDS:
        return False
    return (
        value["action"] in ALLOWED_ACTIONS
        and value["resource"] in ALLOWED_RESOURCES
        and isinstance(value["target"], str)
        and isinstance(value["message"], str)
        and isinstance(value["memory"], str)
        and isinstance(value["amount"], (int, float))
        and not isinstance(value["amount"], bool)
        and math.isfinite(value["amount"])
        and 0 <= value["amount"] <= 2
    )

eval_rows = list(dataset["test"])
rng = random.Random(SEED)
action_to_rows = {}
for row in dataset["test"]:
    action = expected_action(row)["action"]
    action_to_rows.setdefault(action, []).append(row)
eval_rows = []
per_class = max(1, EVAL_EXAMPLES // len(action_to_rows))
for action in sorted(action_to_rows):
    candidates = action_to_rows[action][:]
    rng.shuffle(candidates)
    eval_rows.extend(candidates[:per_class])
eval_rows = eval_rows[:EVAL_EXAMPLES]
print(f"Evaluating {len(eval_rows)} held-out examples across {len(action_to_rows)} actions.")

def evaluate_policy(rows, label):
    records = []
    for index, row in enumerate(rows, 1):
        raw = generate_one(row)
        parsed, strict = parse_one_json(raw)
        target = expected_action(row)
        valid = validate_predicted_action(parsed, strict)
        records.append({
            "raw": raw, "prediction": parsed, "target": target,
            "strict_json": strict, "schema_valid": valid,
            "action_match": valid and parsed["action"] == target["action"],
            "exact_match": valid and parsed == target,
        })
        if index % 25 == 0:
            print(f"{label}: {index}/{len(rows)}")
    n = len(records)
    metrics = {
        "examples": n,
        "strict_single_json_rate": sum(r["strict_json"] for r in records)/n,
        "valid_action_schema_rate": sum(r["schema_valid"] for r in records)/n,
        "action_class_accuracy": sum(r["action_match"] for r in records)/n,
        "exact_six_field_match_rate": sum(r["exact_match"] for r in records)/n,
    }
    return metrics, records

base_metrics, base_records = evaluate_policy(eval_rows, "Base")
print("Base model:", json.dumps(base_metrics, indent=2))


In [ ]:
# Evaluate the untouched base model above, then attach fresh trainable adapters.
FastLanguageModel.for_training(model)
target_modules = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=target_modules,
    lora_alpha=LORA_RANK,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    max_seq_length=MAX_SEQ_LENGTH,
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable LoRA parameters: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")


## 7. Supervised fine-tuning on assistant actions

TRL recognizes the conversational prompt-completion format and computes loss on the **completion only**. Training minimizes next-token cross entropy for the target action JSON.

In [ ]:
from trl import SFTConfig, SFTTrainer

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
args = SFTConfig(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    max_length=MAX_SEQ_LENGTH,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUMULATION,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.01,
    optim="adamw_8bit",
    logging_steps=20,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    completion_only_loss=True,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    gradient_checkpointing=True,
    max_grad_norm=1.0,
    seed=SEED,
    data_seed=SEED,
    dataset_num_proc=1,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    args=args,
)

print(f"Training rows: {len(dataset['train']):,}")
print(f"Validation rows: {len(dataset['validation']):,}")
print(f"Estimated optimizer steps: {math.ceil(len(dataset['train'])/(PER_DEVICE_BATCH_SIZE*GRAD_ACCUMULATION))*NUM_EPOCHS:,}")


In [ ]:
training_result = trainer.train()
print(training_result)
print("Best checkpoint:", trainer.state.best_model_checkpoint)


## 8. Evaluate the fine-tuned adapter on the same held-out examples

A higher action accuracy means more test observations received the teacher's action class. A higher schema-valid rate means more generations are executable JSON at the format level. Physical preconditions still need checking against the current simulator state at inference time.

In [ ]:
FastLanguageModel.for_inference(model)
tuned_metrics, tuned_records = evaluate_policy(eval_rows, "Fine-tuned")
comparison = {"base": base_metrics, "fine_tuned": tuned_metrics}
print(json.dumps(comparison, indent=2))

print("\nExamples")
for index in range(min(5, len(tuned_records))):
    record = tuned_records[index]
    print(json.dumps({
        "input_user_message": eval_rows[index]["prompt"][-1]["content"][:600],
        "target": record["target"],
        "prediction": record["prediction"],
        "raw_generation": record["raw"],
    }, ensure_ascii=False, indent=2))


## 9. Save the adapter and tokenizer

This saves the LoRA adapter and training/evaluation metadata. The base open-weight checkpoint is still required to load the adapter. Optional merged 16-bit export is off by default and requires additional disk space.

In [ ]:
ADAPTER_DIR = OUTPUT_DIR / "adapter"
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))

report = {
    "base_model": MODEL_ID,
    "dataset_manifest": manifest,
    "train_data_sampling": {
        "fraction": TRAIN_DATA_FRACTION,
        "seed": TRAIN_SAMPLE_SEED,
        "strategy": "seeded action-stratified sample of train split",
        "validation_and_test_fraction": 1.0,
    },
    "training": {
        "epochs": NUM_EPOCHS,
        "max_seq_length": MAX_SEQ_LENGTH,
        "batch_size_per_device": PER_DEVICE_BATCH_SIZE,
        "gradient_accumulation_steps": GRAD_ACCUMULATION,
        "lora_rank": LORA_RANK,
        "learning_rate": LEARNING_RATE,
        "train_rows_after_length_filter": len(dataset["train"]),
        "validation_rows_after_length_filter": len(dataset["validation"]),
        "test_rows_after_length_filter": len(dataset["test"]),
        "best_checkpoint": trainer.state.best_model_checkpoint,
    },
    "metrics": comparison,
    "interpretation": "Imitation and formatting metrics for synthetic scenario-teacher labels; not biological validation.",
}
(OUTPUT_DIR / "training_report.json").write_text(json.dumps(report, indent=2))
print(f"Saved adapter: {ADAPTER_DIR}")
print(f"Saved report: {OUTPUT_DIR / 'training_report.json'}")

if SAVE_MERGED_16BIT:
    merged_dir = OUTPUT_DIR / "merged_16bit"
    model.save_pretrained_merged(str(merged_dir), tokenizer, save_method="merged_16bit")
    print(f"Saved merged 16-bit model: {merged_dir}")


## 10. Use the adapter for one simulator-style decision

At runtime, give the model the same system instruction and local JSON observation used in training. The generated text still needs strict JSON parsing and physical action validation before the simulator applies it. Do not send the observer's hive-state vector to the model.

In [ ]:
from bee_local_qwen_policy import BeeLocalQwenPolicy

# This policy implements the simulator's decide(observation) interface locally.
# It can be passed to Simulation(world, local_policy) in a runtime that has
# beehive_sim.py and bee_local_qwen_policy.py beside the notebook.
local_policy = BeeLocalQwenPolicy(model, tokenizer, max_new_tokens=MAX_NEW_TOKENS)
example_observation = json.loads(eval_rows[0]["prompt"][1]["content"])
action = local_policy.decide(example_observation)
print("Local policy action:", json.dumps(action, ensure_ascii=False))
print("Policy label:", local_policy.label)
print("No API request was made.")
print("Adapter directory:", OUTPUT_DIR / "adapter")


## References and data provenance

- [Unsloth Qwen 3.5 fine-tuning guide](https://unsloth.ai/docs/models/qwen3.5/fine-tune): Qwen 3.5 support, Transformers v5 requirement, bf16/16-bit LoRA recommendation and SFTTrainer example.
- [Official Qwen 3.5 0.8B model repository](https://huggingface.co/Qwen/Qwen3.5-0.8B).
- [Unsloth Qwen 3.5 0.8B Colab notebook catalog](https://unsloth.ai/docs/get-started/unsloth-notebooks).
- [TRL SFTTrainer dataset formats](https://huggingface.co/docs/trl/sft_trainer): conversational prompt-completion inputs and completion-only loss.

The full dataset was generated from beehive_sim.World.observe(); its target proposals are checked by World.execute() against the corresponding state. The generator uses deterministic, balanced scenario recipes. Read TRAINING_DATA.md before interpreting training results.